
# 01 — Oracle Database: Queries & Exploration

The SDGFT Oracle Database contains **61.7 million** pre-computed parameter points
covering Δ ∈ [0.200, 0.220] × δ_g ∈ [0.040, 0.043], each with 37 GNN-predicted
observables and χ² scores against 21 experimental measurements.

This notebook shows how to:
1. Load and inspect the database
2. Run pandas queries and filters
3. Use DuckDB for SQL-style analytics
4. Export subsets for external tools

> **Oracle Database DOI:** [10.5281/zenodo.18863347](https://doi.org/10.5281/zenodo.18863347)

**See also:** [docs/oracle_schema.md](../docs/oracle_schema.md) for the full column reference.


In [ ]:
from sdgft_ml.inference import OracleDB
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

db = OracleDB()
print(db.summary())

## 1. Basic Queries

In [ ]:
# Column overview
print(f"Columns ({len(db.columns)}):")
for i, col in enumerate(db.columns):
    print(f"  {i:2d}. {col}", end="\n" if (i + 1) % 3 == 0 else "\t\t")
print()

In [ ]:
# Best-fit points (lowest χ²)
top20 = db.best_fit(20)
display(top20[["delta", "delta_g", "total_chi2", "chi2_per_dof",
               "higgs_mass", "n_s", "omega_m", "sin2_theta_w", "alpha_s"]])

In [ ]:
# Filter by observable range — Higgs mass within 2σ of experiment
higgs_range = db.filter_observable("higgs_mass", 124.91, 125.59)  # 125.25 ± 2×0.17
print(f"Points with m_H ∈ [124.91, 125.59] GeV: {len(higgs_range):,}")
print(f"Best χ² in this subset: {higgs_range['total_chi2'].min():.2f}")

In [ ]:
# Compound query with pandas expression
tight = db.query(
    "higgs_mass > 124.5 and higgs_mass < 126.0 "
    "and n_s > 0.96 and n_s < 0.97 "
    "and chi2_per_dof < 1.5"
)
print(f"Points passing tight cuts: {len(tight):,}")
tight[["delta", "delta_g", "total_chi2", "higgs_mass", "n_s"]].describe()

## 2. Gold Standard Subset

The Gold Standard contains all points with χ²/dof < 1.2 — about 35M rows.

In [ ]:
gold = db.gold_standard()
print(f"Gold Standard rows: {len(gold):,}")
print(f"χ²/dof range: [{gold['chi2_per_dof'].min():.4f}, {gold['chi2_per_dof'].max():.4f}]")
print(f"\nParameter ranges in Gold:")
for col in ["delta", "delta_g"]:
    print(f"  {col}: [{gold[col].min():.6f}, {gold[col].max():.6f}]")

## 3. Statistical Profiles

In [ ]:
# Distribution of key observables in the Gold Standard
obs_highlight = ["higgs_mass", "n_s", "omega_m", "sin2_theta_w", "alpha_s", "r_tensor"]
exp_values = {
    "higgs_mass": 125.25, "n_s": 0.9649, "omega_m": 0.3153,
    "sin2_theta_w": 0.23122, "alpha_s": 0.1180, "r_tensor": 0.0,
}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, obs in zip(axes.flat, obs_highlight):
    vals = gold[obs].values
    ax.hist(vals, bins=100, alpha=0.7, color="steelblue", density=True)
    if obs in exp_values:
        ax.axvline(exp_values[obs], color="red", ls="--", lw=1.5, label="Experiment")
    ax.set_title(obs)
    ax.legend(fontsize=8)

fig.suptitle("Observable Distributions in the Gold Standard (χ²/dof < 1.2)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# χ² distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Full database
ax1.hist(db.df["chi2_per_dof"].values, bins=200, range=(0, 5), alpha=0.7)
ax1.set_xlabel("χ²/dof")
ax1.set_ylabel("Count")
ax1.set_title("Full Database")
ax1.axvline(1.0, color="red", ls="--", label="χ²/dof = 1")
ax1.axvline(1.2, color="orange", ls="--", label="Gold cut")
ax1.legend()

# Gold zoom
ax2.hist(gold["chi2_per_dof"].values, bins=200, range=(0, 1.2), alpha=0.7, color="gold")
ax2.set_xlabel("χ²/dof")
ax2.set_ylabel("Count")
ax2.set_title("Gold Standard")
ax2.axvline(1.0, color="red", ls="--", label="χ²/dof = 1")
ax2.legend()

plt.tight_layout()
plt.show()

## 4. DuckDB SQL Analytics (Optional)

For complex aggregations, DuckDB can query Parquet files directly
without loading them into memory.

Install: `pip install duckdb`

In [ ]:
try:
    import duckdb

    parquet_path = str(db.parquet_path)

    # Binned statistics: mean Higgs mass by Δ bin
    result = duckdb.sql(f"""
        SELECT
            round(delta, 4) AS delta_bin,
            count(*) AS n,
            avg(higgs_mass) AS avg_higgs,
            min(total_chi2) AS min_chi2,
            avg(chi2_per_dof) AS avg_chi2_dof
        FROM read_parquet('{parquet_path}')
        WHERE chi2_per_dof < 1.5
        GROUP BY delta_bin
        ORDER BY delta_bin
        LIMIT 30
    """)
    display(result.df())

except ImportError:
    print("DuckDB not installed — skipping SQL examples.")
    print("Install with: pip install duckdb")

In [ ]:
try:
    import duckdb

    # Correlation query: which observables correlate most with χ²?
    obs_cols = [c for c in db.columns if c not in (
        "delta", "delta_g", "total_chi2", "chi2_per_dof",
        "n_tensions", "gold_standard", "desi_w_match"
    )]

    corr_sql = ", ".join([f"corr({col}, total_chi2) AS corr_{col}" for col in obs_cols[:10]])
    result = duckdb.sql(f"""
        SELECT {corr_sql}
        FROM read_parquet('{parquet_path}')
    """)
    corr_df = result.df().T
    corr_df.columns = ["corr_with_chi2"]
    corr_df.index = [c.replace("corr_", "") for c in corr_df.index]
    display(corr_df.sort_values("corr_with_chi2", key=abs, ascending=False))

except ImportError:
    pass

## 5. Export Subsets

In [ ]:
# Export a filtered subset to CSV
subset = db.query("chi2_per_dof < 1.0")[["delta", "delta_g", "total_chi2",
                                           "higgs_mass", "n_s", "omega_m"]]
print(f"Exporting {len(subset):,} rows with χ²/dof < 1.0")

# Uncomment to save:
# subset.to_csv("oracle_subset_chi2lt1.csv", index=False)
# subset.to_parquet("oracle_subset_chi2lt1.parquet", index=False)

subset.head(10)

---

**Next:** [02 Parameter Landscape](02_parameter_landscape.ipynb) — Sensitivity maps and correlation analysis